# Solution C2 · Group leakage: the same patient in training and in validation

**Type**: Break the model

## Objective

Measure the effect of repeated measurements on the performance estimate: compare `KFold` and `GroupKFold` for a flexible model
and for a rigid one, and show that the model "recognizes" the patient from their measurements.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

p = ds.patient_measurements()
X, y, g = p[["m1", "m2", "m3"]], p["outcome"], p["patient"]
kf = KFold(5, shuffle=True, random_state=0)
gkf = GroupKFold(5)

## Commented solution

In [ ]:
models = {"random forest": RandomForestRegressor(100, random_state=0), "linear regression": LinearRegression()}
rows = {}
for name, m in models.items():
    random_r2 = cross_val_score(m, X, y, cv=kf, scoring="r2").mean()
    group_r2 = cross_val_score(m, X, y, cv=gkf, groups=g, scoring="r2").mean()
    rows[name] = {"random": random_r2, "group": group_r2, "difference": random_r2 - group_r2}
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
def shared_share(splitter, groups=None):
    shares = [np.isin(g.iloc[va], g.iloc[tr]).mean() for tr, va in splitter.split(X, y, groups)]
    return float(np.mean(shares))


kfold_shared_share = shared_share(kf)
group_shared_share = shared_share(gkf, g)
patient_accuracy = cross_val_score(KNeighborsClassifier(1), X, g, cv=kf).mean()
print(f"validation rows whose patient is in the training set: KFold {kfold_shared_share:.2f} | GroupKFold {group_shared_share:.2f}")
print(f"recognition of the patient from the measurements: {patient_accuracy:.3f}")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["random", "group", "difference"] and set(table.index) == {"random forest", "linear regression"},
          "table with the two models and the three columns", "columns: random, group, difference")
    check(table.loc["random forest", "random"] > 0.6 and table.loc["random forest", "group"] < 0,
          "the random forest looks great with random K-fold and becomes worse than the mean with GroupKFold", "random > 0.6, group < 0 for the random forest")
    check(table.loc["random forest", "difference"] > 4 * table.loc["linear regression", "difference"] > 0,
          "the effect of the leakage is much larger for the flexible model (over 4 times that of the linear model)",
          "the forest must have a random-group difference much larger than that of the linear regression")
    check(kfold_shared_share > 0.99 and group_shared_share == 0, "K-fold: almost all the validation patients are also in the training set; GroupKFold: never",
          "with KFold the patient appears in the training set; with GroupKFold the groups are disjoint")
    check(patient_accuracy > 0.8, "the measurements identify the patient with over 80% accuracy (against 1.7% by chance)", "the three measurements are a fingerprint of the patient")
done()

## What you learned

- With random `KFold` every validation row has 8 "twins" (the same patient) in the training set. A random forest recognizes them from the
  measurements and remembers their outcome: R² about 0.75. With `GroupKFold` the validation patients are new and the R² is negative: the model
  has learned nothing transferable. It is the correct estimate if new patients arrive in production.
- The linear regression loses much less (from 0.06 to −0.20, a difference of 0.26 against 1.52): it does not have the capacity to memorize, so
  the leakage helps it little. **Group leakage inflates mostly the flexible models**: a comparison between models done with the
  wrong splitter picks the model that memorizes best, not the one that generalizes best.
- The measurements are a fingerprint: a simple "nearest neighbor" rule recognizes the patient in 88% of the cases (against 1.7% by chance). Every time the rows
  have an implicit identifier (customer, patient, device) the right question is: "will the model see new entities?".

## Common pitfall

Not having a `patient` column ("it is not needed as a feature") and so not thinking about grouping. The group must not be a
feature: it is needed only by the splitter. If you don't pass it, the validation is silently optimistic.

## Interview question

<details><summary>You have 5,000 transactions from 200 customers and you want to evaluate a fraud model that will have to work on new customers. How do you validate?</summary>

With `GroupKFold` (or `StratifiedGroupKFold` if the positive class is rare) grouping by customer, so that all the transactions of a
customer are in the same fold. A random K-fold would put transactions of the same customer in training and in validation and the
model could recognize the customer, inflating the estimate. I also check that the estimate with the groups is consistent with what I see
on truly new customers in production.

</details>